# Feature engineering - data preparation pipeline  | Sebislaw

## Libraries

In [3]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
import xgboost as xgb
from xgboost import XGBClassifier
import optuna

## Data

In [4]:
data_path = '..\\..\\data'

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

## Data preparation pipeline

In [5]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False):

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data  = prepare_data(tourney_results)
    
    # data frame with mean game statistics for a given team in a given season
    season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
        [
            'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
            'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
            'PointDiff'
        ]
    ].agg('mean').reset_index()
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics .copy()
    season_statistics_T2 = season_statistics .copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    
    # data frame containing game's result
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>118].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>118].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_x_y(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            other_parameters=0):
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds)
    
    # Prepare data and labels
    x = df[list(df.columns[7:999])].values
    y = np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    
    return df, x, y

def get_final_data(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage1):

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage1.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(int)
    sample_submission['Team1'] = sample_submission['Team1'].astype(int)
    sample_submission['Team2'] = sample_submission['Team2'].astype(int)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
    # Merge the two DataFrames using a left join on the keys:
    # # For sample_submission: keys are Season, Team1, Team2.
    # # For regular_data_final: keys are Season, T1_TeamID, T2_TeamID.
    # final_regular_season_with_rows_as_in_submission = pd.merge(
    #     sample_submission,
    #     regular_data_final,
    #     left_on=['Season', 'Team1', 'Team2'],
    #     right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
    #     how='left'
    # )
    # final_tourney_data_with_rows_as_in_submission = pd.merge(
    #     sample_submission,
    #     tourney_data_final,
    #     left_on=['Season', 'Team1', 'Team2'],
    #     right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
    #     how='left'
    # )

    # tourney_data =  get_data(final_regular_season_with_rows_as_in_submission, final_tourney_data_with_rows_as_in_submission,
    #                          seeds, prepared=True)

    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True)

    # Prepare data and labels
    x = tourney_data[list(tourney_data.columns[7:999])].values
    y = np.where(
        tourney_data[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(tourney_data['T1_Score'] - tourney_data['T2_Score'] > 0, 1, 0)
    )

    return tourney_data, x, y

## Prepare train and test data set

In [6]:
regular_results = pd.concat([
    MRegularSeasonDetailedResults.copy(),
    WRegularSeasonDetailedResults.copy()
], ignore_index=True)
tourney_results = pd.concat([
    MNCAATourneyDetailedResults.copy(),
    WNCAATourneyDetailedResults.copy()
], ignore_index=True)
seeds = pd.concat([
    MNCAATourneySeeds.copy(),
    WNCAATourneySeeds.copy()
], ignore_index=True)

# PARAMETERS TO CHANGE
# ----------------------------------------------------------
final_season = 2023
start_season = 2005
season_years_list  = [[i-2, i-1, i] for i in range(start_season, final_season+1)]
days_back = 30
# ----------------------------------------------------------

# This ensures that we simulate the scenarion in competition
tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
tourney_results = tourney_results[tourney_results['Season'] < final_season]
tourney_years_list = [[i] for i in range(start_season, final_season+1)]

data = []
x = []
y = []
data_final = []
x_final_season = []
y_final_season = []
for season_years, tourney_years in zip(season_years_list, tourney_years_list):
    if tourney_years[0] != 2020:
        if tourney_years[0] == final_season:
            data_tmp_final, x_tmp, y_tmp = get_final_data(seeds,
                   regular_results, season_years, days_back,
                   tourney_results_final, tourney_years, 
                   SampleSubmissionStage1)
            data_final.append(data_tmp_final)
            x_final_season.append(x_tmp)
            y_final_season.append(y_tmp)
        else:
            data_tmp, x_tmp, y_tmp = get_x_y(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years)
            data.append(data_tmp)
            x.append(x_tmp)
            y.append(y_tmp)

# Concatenate numpy arrays along the first axis
x = np.concatenate(x, axis=0)
y = np.concatenate(y, axis=0)
x_final_season = np.concatenate(x_final_season, axis=0)
y_final_season = np.concatenate(y_final_season, axis=0)
        
# Split data frame into train and test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)
df = pd.concat(data, ignore_index=True)
df_final = pd.concat(data_final, ignore_index=True)

In [15]:
pd.DataFrame(x).to_csv('X.csv', index=False)
pd.DataFrame(y).to_csv('y.csv', index=False)
pd.DataFrame(x_final_season).to_csv('X_final_season.csv', index=False)
pd.DataFrame(y_final_season).to_csv('y_final_season.csv', index=False)

In [18]:
X = pd.read_csv('X.csv').to_numpy()
y = pd.read_csv('y.csv').to_numpy()
x_final_season = pd.read_csv('X_final_season.csv').to_numpy()
y_final_season = pd.read_csv('y_final_season.csv').to_numpy()

## First models

In [21]:
# Create a pipeline with StandardScaler and LogisticRegressionCV
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegressionCV(
        cv=5,                # 5-fold cross-validation
        solver='lbfgs',
        max_iter=2000,       # Increase max_iter if needed
        scoring='neg_log_loss',  # Optimize log-loss for probability calibration
        refit=True
    ))
])

# Train the model using the pipeline
pipeline.fit(x, y)

# Predict probabilities for the positive class
# Create masks for training data:
mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
x_clean = x[mask_train]
y_clean = y[mask_train]
# And for the final season (test) data:
mask_test = ~np.isnan(x_final_season).any(axis=1) & ~np.isnan(y_final_season)
x_final_clean = x_final_season[mask_test]
y_final_clean = y_final_season[mask_test]
# Predict probabilities for the positive class on the cleaned final season data
y_prob = pipeline.predict_proba(x_final_clean)[:, 1]

# Calculate the Brier score using the cleaned test labels
score = brier_score_loss(y_final_clean, y_prob)
print("Brier score:", score)

Brier score: 0.1787828921363371


In [22]:
# Create and train the XGBoost classifier
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model.fit(x, y)

# Predict probabilities for the positive class
# Create masks for training data:
mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
x_clean = x[mask_train]
y_clean = y[mask_train]
# And for the final season (test) data:
mask_test = ~np.isnan(x_final_season).any(axis=1) & ~np.isnan(y_final_season)
x_final_clean = x_final_season[mask_test]
y_final_clean = y_final_season[mask_test]
# Predict probabilities for the positive class on the cleaned final season data
y_prob = model.predict_proba(x_final_clean)[:, 1]

# Calculate the Brier score
brier = brier_score_loss(y_final_clean, y_prob)
print("Brier Score:", brier)

c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:00:31] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Brier Score: 0.21200261550085395


In [26]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Define the base models
base_models = [
    ('lr', LogisticRegressionCV(
        cv=5,                # 5-fold cross-validation
        solver='lbfgs',
        max_iter=2000,       # Increase max_iter if needed
        scoring='neg_log_loss',  # Optimize log-loss for probability calibration
        refit=True
    )),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss')),
    ('svc', SVC(probability=True)),
    ('knn', KNeighborsClassifier()),
    ('dt', DecisionTreeClassifier()),
    ('rf', RandomForestClassifier())
]

# Define the meta-model
meta_model = LogisticRegression()

# Create the stacking classifier
stacking_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)

# Train the stacking classifier
stacking_clf.fit(x, y)

# Predict probabilities for the positive class on the cleaned final season data
y_prob_stacking = stacking_clf.predict_proba(x_final_clean)[:, 1]

# Calculate the Brier score using the cleaned test labels
brier_score_stacking = brier_score_loss(y_final_clean, y_prob_stacking)
print("Brier score (stacking):", brier_score_stacking)

c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:03:07] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:03:20] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\nextcloud\Studia - PW\semestr 6\projekt interdyscyplinarny\march-ml-madness-2025\.venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:03:21] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08

Brier score (stacking): 0.17945143505772346


# Models with optimization

## Logistic Regression

In [30]:
pip install scikit-optimize

Note: you may need to restart the kernel to use updated packages.


In [5]:
from skopt import BayesSearchCV
from sklearn.svm import SVC
from sklearn.metrics import brier_score_loss

# Define a custom scorer function.
# It calls predict_proba and returns the negative Brier score.
def custom_brier_scorer(estimator, X, y):
    y_prob = estimator.predict_proba(X)[:, 1]
    # Negative because lower Brier score is better.
    return -brier_score_loss(y, y_prob)

# Set up BayesSearchCV with detailed progress reporting via verbose.
opt = BayesSearchCV(
    SVC(probability=True),  # enable probability estimates
    {
        'C': (1e-6, 1e+6, 'log-uniform'),
        'gamma': (1e-6, 1e+1, 'log-uniform'),
        'degree': (1, 8),            # integer-valued parameter
        'kernel': ['linear', 'poly', 'rbf']  # categorical parameter
    },
    n_iter=32,
    cv=3,
    scoring=custom_brier_scorer,
    verbose=3,  # This will print progress, ETA, etc.
    n_jobs=6
)

# Assume X_train, y_train are your training data, and X_test, y_test are your test data.
opt.fit(x, y)

# Use the best estimator to predict probabilities on the test set.
y_prob = opt.best_estimator_.predict_proba(x_final_clean)[:, 1]

# Calculate the Brier score on the test set.
score = brier_score_loss(y_final_clean, y_prob)
print("Brier score:", score)


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits


KeyboardInterrupt: 

# GPU

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

In [11]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import KFold

# Example PyTorch model: a simple linear classifier with sigmoid activation.
class LinearClassifier(nn.Module):
    def __init__(self, input_dim):
        super(LinearClassifier, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
    
    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# Function to train the model for one epoch.
def train_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    for X_batch, y_batch in data_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).view(-1, 1)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

# Function to evaluate the model and return Brier score and predicted probabilities.
def evaluate(model, X, y, device):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
        y_prob = model(X_tensor).cpu().numpy().flatten()
    score = brier_score_loss(y, y_prob)
    return score, y_prob

# Assume that x, y, x_final_season, y_final_season are already defined numpy arrays.
# Clean training data: remove rows with NaN in features or labels.
mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
x_clean = x[mask_train]
y_clean = y[mask_train]

# Clean test (final season) data.
mask_test = ~np.isnan(x_final_season).any(axis=1) & ~np.isnan(y_final_season)
x_final_clean = x_final_season[mask_test]
y_final_clean = y_final_season[mask_test]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_dim = x_clean.shape[1]

# New hyperparameter grid based on previous results.
learning_rates = [0.0005, 0.001, 0.002]
epoch_options = [20, 30, 40]

best_cv_score = float('inf')
best_params = {}
best_state = None

# 3-fold cross-validation using KFold on the cleaned training data.
kf = KFold(n_splits=3, shuffle=True, random_state=42)

for lr in learning_rates:
    for num_epochs in epoch_options:
        cv_scores = []
        for train_idx, val_idx in kf.split(x_clean):
            X_train, X_val = x_clean[train_idx], x_clean[val_idx]
            y_train, y_val = y_clean[train_idx], y_clean[val_idx]
            
            # Initialize model, criterion and optimizer.
            model = LinearClassifier(input_dim).to(device)
            criterion = nn.BCELoss()
            optimizer = optim.SGD(model.parameters(), lr=lr)
            
            # Create a DataLoader for training.
            train_dataset = torch.utils.data.TensorDataset(
                torch.tensor(X_train, dtype=torch.float32), 
                torch.tensor(y_train, dtype=torch.float32)
            )
            train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
            
            # Train the model.
            for epoch in range(num_epochs):
                train_epoch(model, train_loader, criterion, optimizer, device)
            
            # Evaluate on the validation set.
            score, _ = evaluate(model, X_val, y_val, device)
            cv_scores.append(score)
        
        avg_score = np.mean(cv_scores)
        print(f"Learning rate: {lr}, Epochs: {num_epochs}, CV Brier Score: {avg_score:.4f}")
        
        if avg_score < best_cv_score:
            best_cv_score = avg_score
            best_params = {'lr': lr, 'num_epochs': num_epochs}
            best_state = model.state_dict()  # from the last fold (approximation)

print("Best hyperparameters found:", best_params)

# Retrain on the full cleaned training set using the best hyperparameters.
best_model = LinearClassifier(input_dim).to(device)
criterion = nn.BCELoss()
optimizer = optim.SGD(best_model.parameters(), lr=best_params['lr'])
train_dataset = torch.utils.data.TensorDataset(
    torch.tensor(x_clean, dtype=torch.float32), 
    torch.tensor(y_clean, dtype=torch.float32)
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

for epoch in range(best_params['num_epochs']):
    train_epoch(best_model, train_loader, criterion, optimizer, device)

# Evaluate on the cleaned test set.
test_score, y_prob = evaluate(best_model, x_final_clean, y_final_clean, device)
print("Test Brier score:", test_score)


Learning rate: 0.0005, Epochs: 20, CV Brier Score: 0.2158
Learning rate: 0.0005, Epochs: 30, CV Brier Score: 0.2071
Learning rate: 0.0005, Epochs: 40, CV Brier Score: 0.2029
Learning rate: 0.001, Epochs: 20, CV Brier Score: 0.2241
Learning rate: 0.001, Epochs: 30, CV Brier Score: 0.3435
Learning rate: 0.001, Epochs: 40, CV Brier Score: 0.3242
Learning rate: 0.002, Epochs: 20, CV Brier Score: 0.3098
Learning rate: 0.002, Epochs: 30, CV Brier Score: 0.5000
Learning rate: 0.002, Epochs: 40, CV Brier Score: 0.5000
Best hyperparameters found: {'lr': 0.0005, 'num_epochs': 40}
Test Brier score: 0.4092848786852087
